# Agent-Enhanced GraphRAG (Colab)
This notebook installs dependencies and runs B1/B2/Ours on HotpotQA.

In [ ]:
%pip install -q -U pip
%pip install -q langgraph langchain langchain-community datasets sentence-transformers faiss-cpu networkx transformers accelerate bitsandbytes python-dotenv typer rich tqdm pydantic scikit-learn pandas evaluate rapidfuzz
!python -m spacy download en_core_web_sm

In [ ]:
import os
os.environ['LLM_BACKEND'] = 'transformers'
os.environ['HF_MODEL_NAME'] = 'meta-llama/Llama-3.2-3B-Instruct'
os.environ['HOTPOT_SUBSET_SIZE'] = '200'
print('CUDA available:', __import__('torch').cuda.is_available())

In [ ]:
from src.utils.data_loader import prepare_hotpotqa_subset
examples = prepare_hotpotqa_subset(subset_size=200)
print('Prepared examples:', len(examples))

In [ ]:
from src.pipeline import AgentEnhancedGraphRAG
from src.utils.data_loader import load_prepared_subset
from src.utils.evaluation import run_method_on_dataset
from dataclasses import asdict

pipe = AgentEnhancedGraphRAG()
data = [asdict(x) for x in load_prepared_subset(200)]
b1 = run_method_on_dataset('B1_Standard_Dense_RAG', data, lambda x: pipe.dense_rag_baseline(x['question'], x['context']))
b2 = run_method_on_dataset('B2_Basic_GraphRAG', data, lambda x: pipe.basic_graphrag_baseline(x['question'], x['context']))
ours = run_method_on_dataset('Ours_Agent_Enhanced_GraphRAG', data, lambda x: pipe.invoke(x['question'], x['context']))
print('B1:', b1['metrics'])
print('B2:', b2['metrics'])
print('OURS:', ours['metrics'])